<a href="https://colab.research.google.com/github/GlobalFishingWatch/gfw-api-python-client/blob/develop/notebooks/workflow-guides/workflow-02-analyze-apparent-fishing-effort-argentinian-eez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analyze apparent fishing effort in Argentinian EEZ

This guide provides detailed instructions to on how to use the [gfw-api-python-client](https://github.com/GlobalFishingWatch/gfw-api-python-client) to **Analyze apparent fishing effort in [Argentinian EEZ](https://www.marineregions.org/gazetteer.php?p=details&id=8466) region and monitor industrial trawlers** using **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)**.

**Note:** See the [Datasets](https://globalfishingwatch.org/our-apis/documentation#api-dataset), [Data Caveats](https://globalfishingwatch.org/our-apis/documentation#data-caveat), and [Terms of Use](https://globalfishingwatch.org/our-apis/documentation#terms-of-use) pages in the [GFW API documentation](https://globalfishingwatch.org/our-apis/documentation#introduction) for details on GFW data, API licenses, and rate limits.

## Prerequisites

Before using the `gfw-api-python-client`, ensure it is installed (see the [Getting Started](https://globalfishingwatch.github.io/gfw-api-python-client/getting-started.html) guide) and that you have obtained an API access token from the [Global Fishing Watch API portal](https://globalfishingwatch.org/our-apis/tokens).

## Installation

The `gfw-api-python-client` can be installed easily from either the [Python Package Index (PyPI)](https://pypi.org/project/gfw-api-python-client/) or [Conda](https://anaconda.org/conda-forge/gfw-api-python-client)

In [1]:
# %pip install gfw-api-python-client

In [2]:
# %conda install -c conda-forge gfw-api-python-client

## Usage

Import and use `gfw-api-python-client` in your Python codes

In [3]:
import datetime
import os

import pandas as pd

import gfwapiclient as gfw

In [4]:
try:
    from google.colab import userdata

    access_token = userdata.get("GFW_API_ACCESS_TOKEN")
except Exception:
    access_token = os.environ.get("GFW_API_ACCESS_TOKEN")

access_token = access_token or "<PASTE_YOUR_GFW_API_ACCESS_TOKEN_HERE>"

In [5]:
gfw_client = gfw.Client(
    access_token=access_token,
)

## Introduction

**Use Case: A Fisheries Enforcement Officer Monitoring Industrial Trawlers**

Maria, a fisheries enforcement officer in Argentina, monitors industrial trawlers operating within **[Argentinian Exclusive Economic Zone (EEZ)](https://www.marineregions.org/gazetteer.php?p=details&id=8466)**. His goal is to:

1. Analyzing apparent fishing effort for **trawlers** operating in Argentinian EEZ.
2. Generating reports to support fisheries enforcement decisions.

**API Used:**
️
1. **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** – Retrieve **apparent fishing effort** data for trawlers.

**Important:** In order to avoid any misinterpretation of **GFW data**, please refer to our official **data caveats** documentations:
- [Apparent fishing effort](https://globalfishingwatch.org/dataset-and-code-fishing-effort/) 
- [Exclusive economic zone boundaries definition](https://globalfishingwatch.org/our-apis/documentation#exclusive-economic-zone-boundaries-definition)

**Important Caveats:**

1. The [4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api) only supports **one active report per user at a time**.
2. **Sending multiple requests simultaneously** results in a **429 Too Many Requests** error.
3. If a report takes over **100 seconds** to generate, it may return a **524 Gateway Timeout** error.

## Step 0: Identify the Region of Interest (ROI) - Argentinian EEZ

Before making API requests, Maria must specify the geographic area for analysis using a **Region ID**:

**Options to Define the Region:**

1. **Using Region ID** - Each EEZ has a unique ID in the **[public-eez-areas](https://globalfishingwatch.org/our-apis/documentation#regions)** dataset.
2. **Custom Geometries** - Users can define a custom area using GeoJSON.
   
For **[Argentinian EEZ, the region ID is 8466](https://www.marineregions.org/gazetteer.php?p=details&id=8466)** (public-eez-areas dataset).

**Note:** See how to use the [Reference Data API - Usage Guides](https://globalfishingwatch.github.io/gfw-api-python-client/usage-guides/references-data-api.html) to obtain and filter predefined [**Regions of Interest (ROIs)**](https://globalfishingwatch.org/our-apis/documentation#regions), such as Exclusive Economic Zones (**EEZs**), Marine Protected Areas (**MPAs**), and Regional Fisheries Management Organizations (**RFMOs**).

In [6]:
eez_rois_result = await gfw_client.references.get_eez_regions(iso3="ARG")
arg_eez_roi = eez_rois_result.data()[0]

In [7]:
arg_eez_roi.id, arg_eez_roi.dataset, arg_eez_roi.label, arg_eez_roi.iso3

('8466', 'public-eez-areas', 'Argentinian Exclusive Economic Zone', 'ARG')

## Step 1: Retrieve Apparent Fishing Effort in Argentinian EEZ

Maria **first queries** the **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** to get **apparent fishing effort for all vessels**, grouping them by **gear type** in **[Argentinian EEZ](https://www.marineregions.org/gazetteer.php?p=details&id=8466)**. Please [learn more about apparent fishing effort here](https://globalfishingwatch.org/our-apis/documentation#ais-apparent-fishing-effort) and [check its data caveats here](https://globalfishingwatch.org/our-apis/documentation#apparent-fishing-effort).

**Filters Used:**

1. **[Region ID](https://globalfishingwatch.org/our-apis/documentation#regions)** - 8466 Argentinian EEZ
2. **[Date Range](https://globalfishingwatch.org/our-apis/documentation#report-url-parameters-for-both-post-and-get-requests)** - Last 6 Months
3. **[Grouped By](https://globalfishingwatch.org/our-apis/documentation#report-url-parameters-for-both-post-and-get-requests)** - Gear Type

**Why This Step?**

- Identifies which gear types (e.g., `trawlers`, `squid jiggers` etc.) are most active in the Argentinian EEZ.
- Establishes baseline fishing activity trends.

In [8]:
end_date = datetime.date.today()

In [9]:
start_date = end_date - datetime.timedelta(weeks=24)

In [10]:
start_date, end_date

(datetime.date(2026, 5, 17), datetime.date(2026, 11, 1))

In [11]:
step_1_report_result = await gfw_client.fourwings.create_fishing_effort_report(
    spatial_resolution="HIGH",
    group_by="GEARTYPE",
    temporal_resolution="MONTHLY",
    start_date=start_date,
    end_date=end_date,
    spatial_aggregation=True,
    region=arg_eez_roi,
)

In [12]:
step_1_report_df = step_1_report_result.df()

In [13]:
step_1_report_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   date                     13 non-null     str    
 1   detections               0 non-null      object 
 2   flag                     0 non-null      object 
 3   gear_type                13 non-null     str    
 4   hours                    13 non-null     float64
 5   vessel_ids               13 non-null     int64  
 6   vessel_id                0 non-null      object 
 7   vessel_type              0 non-null      object 
 8   entry_timestamp          0 non-null      object 
 9   exit_timestamp           0 non-null      object 
 10  first_transmission_date  0 non-null      object 
 11  last_transmission_date   0 non-null      object 
 12  imo                      0 non-null      object 
 13  mmsi                     0 non-null      object 
 14  call_sign                0 non-null    

In [14]:
step_1_report_df[["gear_type", "hours", "vessel_ids"]].head()

,gear_type,hours,vessel_ids
0,trawlers,29762.064444,248
1,trawlers,31735.364167,167
2,trawlers,59481.128056,303
3,fishing,532.781389,8
4,fishing,330.725833,6


In [15]:
step_1_agg_report_df = (
    step_1_report_df.groupby(["gear_type"], as_index=False)
    .agg(hours=("hours", "sum"), vessel_ids=("vessel_ids", "sum"))
    .sort_values(by="hours", ascending=False)
)

In [16]:
step_1_agg_report_df.head()

,gear_type,hours,vessel_ids
2,trawlers,266216.206389,1507
0,fishing,1683.905000,33
1,fixed_gear,2.195556,1


### What We have Learned from Step 1

- Multiple **gear types** were potentially detected in Argentinian EEZ.
- `Trawlers` appear to be operating in the area.

### Summary of API Flow

1. **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** - Retrieve apparent fishing effort for **trawlers** within Argentinian EEZ.
2. **Assess potential risks** - Compare apparent fishing activity against national or RFMO regulations for enforcement follow-ups.
3. **Generate a report** - Provide a structured analysis for relevant authorities.